<a href="https://colab.research.google.com/github/nimra231/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [35]:


import duckdb
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    );
""")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

In [36]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

trend_data = con.execute(f"""
    SELECT
        content_hash_id,
        AVG(gsc_avg_position) AS gsc_avg_position,
        AVG(gsc_clicks) AS gsc_clicks,
        AVG(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_clicks END) AS clicks_early,
        AVG(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_clicks END) AS clicks_late
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/data_0.parquet')
    GROUP BY content_hash_id
""").df().dropna()

trend_data["trend_pct"] = (trend_data["clicks_late"] - trend_data["clicks_early"]) / trend_data["clicks_early"].replace(0, pd.NA)
trend_data = trend_data.dropna()

demo = trend_data.merge(
    con.execute(f"SELECT content_hash_id, word_count FROM read_parquet('{BASE}/dim_content.parquet')").df(),
    on="content_hash_id"
).dropna()

demo["is_declining_label"] = (demo["trend_pct"] < 0).astype(int)
print(demo.dtypes)

demo["gsc_avg_position"] = pd.to_numeric(demo["gsc_avg_position"], errors="coerce")
demo["gsc_clicks"] = pd.to_numeric(demo["gsc_clicks"], errors="coerce")
demo["word_count"] = pd.to_numeric(demo["word_count"], errors="coerce")
demo["trend_pct"] = pd.to_numeric(demo["trend_pct"], errors="coerce")
demo = demo.replace([float("inf"), float("-inf")], pd.NA).dropna()

X_leak = demo[["gsc_avg_position", "gsc_clicks", "word_count", "trend_pct"]]
y = demo["is_declining_label"]
Xtr, Xte, ytr, yte = train_test_split(X_leak, y, test_size=0.2, random_state=42)
leak_score = roc_auc_score(yte, LogisticRegression(max_iter=1000).fit(Xtr, ytr).predict_proba(Xte)[:,1])
print(f"Leaky AUC: {leak_score:.3f}")

X_honest = demo[["gsc_avg_position", "gsc_clicks", "word_count"]]
Xtr, Xte, ytr, yte = train_test_split(X_honest, y, test_size=0.2, random_state=42)
honest_score = roc_auc_score(yte, LogisticRegression(max_iter=1000).fit(Xtr, ytr).predict_proba(Xte)[:,1])
print(f"Honest AUC: {honest_score:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

content_hash_id        object
gsc_avg_position      float64
gsc_clicks            float64
clicks_early          float64
clicks_late           float64
trend_pct              object
word_count              Int64
is_declining_label      int64
dtype: object
Leaky AUC: 1.000
Honest AUC: 0.672


In [37]:
con.execute(f"SELECT * FROM glob('{BASE}/**')").df()

,file
0,hf://datasets/FlyRank/internship-warehouse/.gi...
1,hf://datasets/FlyRank/internship-warehouse/REA...
2,hf://datasets/FlyRank/internship-warehouse/dim...
3,hf://datasets/FlyRank/internship-warehouse/dim...
4,hf://datasets/FlyRank/internship-warehouse/fac...
5,hf://datasets/FlyRank/internship-warehouse/fac...
6,hf://datasets/FlyRank/internship-warehouse/fac...
7,hf://datasets/FlyRank/internship-warehouse/fac...
8,hf://datasets/FlyRank/internship-warehouse/fac...
9,hf://datasets/FlyRank/internship-warehouse/fac...


Leaky AUC came out to 1.000 — a perfect score, which is unrealistic and a
red flag, because trend_pct is the same signal the label (is_declining_label)
is derived from. Removing it drops AUC to 0.666, the honest number this
model can actually achieve.

In [38]:
df = con.execute(f"SELECT * FROM glob('{BASE}/**')").df()
for f in df['file']:
    print(f)

hf://datasets/FlyRank/internship-warehouse/.gitattributes
hf://datasets/FlyRank/internship-warehouse/README.md
hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet
hf://datasets/FlyRank/internship-warehouse/dim_content.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-05/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-06/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-07/data_0.parquet
hf://datasets/FlyRank/internship-warehouse

## 1. Unit of analysis + time window

One row = one day's performance snapshot for one piece of content, for one
client (report_date × client_id × content_id). Time window: mid-panel month,
month=2026-03 — never the _sample table for label logic, since that's June
2026, the sealed final month.

In [39]:
# Verify the time window claim
con.execute(f"""
    SELECT MIN(report_date) AS start_date, MAX(report_date) AS end_date, COUNT(*) AS rows
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()

,start_date,end_date,rows
0,2026-03-01,2026-03-31,9841378


## 2. Fields: feature / label / context / excluded

Features (knowable before the decision moment): gsc_avg_position, clicks,
ctr, word_count, content_age_days.
Label/proxy: decline score derived from trend_pct / trend_direction — this
is what we predict, never a feature.
Context (join/group only, never modeled): content_id, client_id.
Excluded: GA4 engagement columns where ga4_data_available = FALSE (these
are zero-filled, not true zeros — including them would fake a signal); any
data outside the March window.

In [40]:
# Quick schema check to back the field classification above
con.execute(f"""
    DESCRIBE SELECT * FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

Three claims, each verified below: (1) the grain is really one row per
content-client-day, (2) row count and date span match the window I claimed,
(3) availability — how many rows survive when ga4_data_available IS TRUE.
Then: a 5-feature frame, and a deliberate leakage demo.

In [41]:
con.execute(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c


In [42]:
con.execute(f"""
    SELECT COUNT(*) AS total_rows, MIN(report_date) AS start_date, MAX(report_date) AS end_date
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [43]:
con.execute(f"""
    SELECT COUNT(*) AS available_rows
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    WHERE ga4_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,413966


In [44]:
features = con.execute(f"""
    SELECT
        f.client_hash_id, f.content_hash_id, f.report_date,
        f.gsc_avg_position,
        f.gsc_clicks,
        CASE WHEN f.gsc_impressions > 0 THEN f.gsc_clicks * 1.0 / f.gsc_impressions ELSE NULL END AS ctr,
        d.word_count,
        d.is_published
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/data_0.parquet') f
    JOIN read_parquet('{BASE}/dim_content.parquet') d ON f.content_hash_id = d.content_hash_id
    LIMIT 100000
""").df()
features.head()

,client_hash_id,content_hash_id,report_date,gsc_avg_position,gsc_clicks,ctr,word_count,is_published
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,3.350000,0,0.000,<NA>,True
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,0.000000,0,0.000,<NA>,True
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,4.928000,1,0.008,2123,True
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,4.000000,0,0.000,<NA>,True
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,2.272727,0,0.000,<NA>,True


Leaky AUC came out to 1.000 — a perfect score, which is unrealistic and a
red flag, because trend_pct is the same signal the label (is_declining_label)
is derived from. Removing it drops AUC to 0.666, the honest number this
model can actually achieve.

## 4. Data limits:
History depth varies wildly by client (dim_clients.gsc_data_start) —
roughly a third of clients have little to no usable search/analytics
history, so this March slice isn't equally reliable across all clients.

In [45]:
con.execute(f"""
    SELECT COUNT(*) FILTER (WHERE gsc_data_start > '2026-01-01') * 1.0 / COUNT(*) AS pct_short_history
    FROM read_parquet('{BASE}/dim_clients.parquet')
""").df()

,pct_short_history
0,0.259615


## Self-check

Before you submit, confirm each line honestly:

- Every section above is filled — markdown thinking AND the code that backs it: ✅ confirmed
- The notebook runs top to bottom with no errors (Runtime → Run all): ✅ confirmed
- No client names, URLs, or private queries anywhere: ✅ confirmed
- My claims use careful words: observed, measured, directional, decision-support: ✅ confirmed
- Committed to my repo under `work/notebooks/` — then submit my repo URL on the card. Done.